# Day 30 · From numbers to yes/no: will this customer escalate?

**First-principles question:** How does a model give a probability instead of a number?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 300 fake customers: days waiting for a fix, complaints in the last 30 days, and did they escalate?
rng = np.random.default_rng(11)
days_waiting = rng.uniform(0, 5, 300).round(1)
complaints = rng.poisson(1.0, 300)
true_z = -4.0 + 1.0 * days_waiting + 0.9 * complaints
escalated = (rng.random(300) < 1 / (1 + np.exp(-true_z))).astype(float)

X = np.column_stack([days_waiting, complaints])
X_train, y_train, X_test, y_test = X[:200], escalated[:200], X[200:], escalated[200:]
print(f"{escalated.mean():.0%} of customers escalated")

## 1 · Sigmoid: any number → a probability

In [ ]:
def sigmoid(z):
    # TODO: 1 / (1 + e^(-z))   (np.exp works on arrays)
    ...

zs = np.linspace(-8, 8, 200)
plt.plot(zs, sigmoid(zs), color="#2153c4"); plt.axhline(0.5, color="#8a9aa1", linestyle="--")
plt.title("sigmoid(z)"); plt.xlabel("z"); plt.ylabel("probability"); plt.show()

## 2 · The model and its loss

In [ ]:
def predict_proba(X, w, b):
    # TODO: sigmoid of (X @ w + b)
    ...

def log_loss(X, y, w, b):
    p = np.clip(predict_proba(X, w, b), 1e-9, 1 - 1e-9)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

## 3 · Train it
For log loss, the gradient is short: `(p − y)` times the input. (The chain rule from day 6 gives it; take it as given today.)

In [ ]:
w, b = np.zeros(2), 0.0
losses = []
for step in range(2000):
    p = predict_proba(X_train, w, b)
    grad_w = X_train.T @ (p - y_train) / len(y_train)
    grad_b = float(np.mean(p - y_train))
    w, b = w - 0.5 * grad_w, b - 0.5 * grad_b
    losses.append(log_loss(X_train, y_train, w, b))
print(f"w = {w.round(2)}, b = {b:.2f}, final loss {losses[-1]:.3f}")
plt.plot(losses); plt.xlabel("step"); plt.ylabel("log loss"); plt.show()

## 4 · Better than 'always no'?

In [ ]:
# TODO: accuracy on the TEST customers: predict 1 when the probability is at least 0.5
accuracy = None
baseline = max(y_test.mean(), 1 - y_test.mean())     # always guess the most common answer
print(f"model {accuracy:.0%} · always guessing the common answer {baseline:.0%}")

# the decision line: where w·x + b = 0
d = np.linspace(0, 5, 50)
plt.scatter(X_test[:, 0], X_test[:, 1] + rng.uniform(-0.15, 0.15, len(X_test)), c=y_test, cmap="coolwarm", s=15)
plt.plot(d, -(w[0] * d + b) / w[1], color="black", label="50% line")
plt.xlabel("days waiting"); plt.ylabel("complaints"); plt.ylim(-0.5, 5); plt.legend(); plt.show()

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert abs(sigmoid(0) - 0.5) < 1e-12 and sigmoid(100) > 0.999 and sigmoid(-100) < 0.001
assert predict_proba(np.array([[0.0, 0.0]]), np.array([1.0, 1.0]), 0.0)[0] == 0.5
assert losses[-1] < losses[0], "The loss must go down"
assert accuracy is not None and accuracy > baseline, "The model should beat always guessing the common answer"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")